In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
from keras.utils import image_dataset_from_directory as LdImg
from keras.layers import Rescaling, RandomFlip, Resizing,Conv2D,MaxPooling2D,Flatten,Dense,Input
from keras.models import Sequential
from keras.saving import save_model, load_model
from matplotlib import pyplot as plt
from sklearn.metrics import accuracy_score
import tensorflow as tf
import numpy as np

In [ ]:
#load the cnn model
TrainedCNNModle = load_model("CNNModel.keras")
TrainedCNNModle.summary()

In [ ]:
# load test images with label
TestData = LdImg("dataset/test_set", labels="inferred",
                  label_mode="int",
                  color_mode="rgb",
                  shuffle = False,
                  batch_size=None,
                  image_size=(256, 256),
                  crop_to_aspect_ratio=True)

In [ ]:
# define preprocessing, rescaling and resizeing
PreProcessing = Sequential([
    #rescales input values in [0 1]
    Rescaling(1./255),
    #resizing the image to 50x50
    Resizing(50,50)
])

# run prerprocessing
TestDataPreProcd = TestData.map(lambda image, label: (PreProcessing(image),label))

In [ ]:
# run prediction with loaded images 
# store the predicted results and labels into a list
PredResults = []
ExpectedResults = []
for image, label in TestDataPreProcd:
    pred = TrainedCNNModle.predict(tf.expand_dims(image, axis=0),verbose=0)
    pred = (pred>0.5).astype(int)
    PredResults.extend(pred[0])
    ExpectedResults.extend(np.array([label.numpy()]))

In [ ]:
# transform the results list to a np array
ComparResults = np.array(list(zip(PredResults, ExpectedResults))).reshape(-1,2)
print(ComparResults.shape)

In [ ]:
# calculate the accuracy
AccuracyScore = accuracy_score(ComparResults[:,1],ComparResults[:,0])
print(AccuracyScore)

In [ ]:
# load unlabeled images
ImgFromInternet = LdImg("dataset/nolabed_cats_dogs", labels=None,
                  label_mode=None,
                  color_mode="rgb",
                  shuffle = False,
                  batch_size=None,
                  image_size=(256, 256),
                  crop_to_aspect_ratio=True)
ImgFromInternetPreProcd = ImgFromInternet.map(lambda image: (PreProcessing(image)))
print(type(ImgFromInternet))

In [ ]:
# plot the unlabeled image with predicted results
figure=plt.figure()
i=int(1)
for image in ImgFromInternetPreProcd:
    pred = TrainedCNNModle.predict(tf.expand_dims(image, axis=0),verbose=0)
    pred = (pred>0.5).astype(int)    
    plt.subplot(4,2,i)
    plt.imshow((image.numpy()*255).astype("uint8"))
    plt.title(f'it is a {"dog" if pred==1 else "cat"}')
    i += 1
plt.show()